## Import Libraries

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import string 
import nltk
from sklearn.linear_model import LogisticRegression
import re
import pickle

## Download import nltk tools 

In [2]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("tagsets_json")

[nltk_data] Downloading package stopwords to C:\Users\Kajal
[nltk_data]     Prajapati\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Kajal
[nltk_data]     Prajapati\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Kajal
[nltk_data]     Prajapati\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Kajal
[nltk_data]     Prajapati\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Kajal Prajapati\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets_json to C:\Users\Kajal
[nltk_data]     Prajapati\AppData\Roam

True

## Data cleaning

In [3]:
df = pd.read_csv("FakeNewsNet.csv")

In [4]:
df.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [5]:
df.isnull().sum()

title              0
news_url         330
source_domain    330
tweet_num          0
real               0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23196 entries, 0 to 23195
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          23196 non-null  object
 1   news_url       22866 non-null  object
 2   source_domain  22866 non-null  object
 3   tweet_num      23196 non-null  int64 
 4   real           23196 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 906.2+ KB


In [7]:
df = df.fillna(" ")

In [8]:
df["real"].value_counts() # 1Real,0Fake 

real
1    17441
0     5755
Name: count, dtype: int64

In [9]:
df["content"] = df["title"]+" "+df["source_domain"]

In [10]:
df["content"] = df["content"].apply(lambda x: re.sub(r"[^ a-zA-Z\s]","",str(x)))

In [11]:
df["content"] = df["content"].str.lower()

In [12]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
def clean_text(text):
    text = text.translate(str.maketrans("","",string.punctuation))
    tokenize = word_tokenize(text)
    filtered_words = [w for w in tokenize if w not in stop_words]

    # I used **lemmetization** instead of **stemming** because on pos it works more accuratly
    lem = WordNetLemmatizer()
    lemma = [lem.lemmatize(w, pos = "v") for w in filtered_words]
    return " ".join(lemma)

df["content"] = df["content"].apply(clean_text)

In [13]:
df["content"][0]

'kandi burruss explode rape accusation real housewives atlanta reunion video toofabcom'

In [14]:
X = df["content"].values
y = df["real"].values

## Vectorization before modeling 

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X)

In [16]:
feature_names = vectorizer.get_feature_names_out()
print(feature_names)

['aaliyah' 'aaliyahs' 'aap' ... 'zwierzynski' 'zylka' 'zyrus']


In [17]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size = 0.2, random_state = 1, stratify = y)
X_train.shape

(18556, 18572)

In [18]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [19]:
y_pred = model.predict(X_test)

In [20]:
from sklearn.metrics import accuracy_score,f1_score,classification_report

acc = accuracy_score(y_pred,y_test)
f1 = f1_score(y_pred,y_test)
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.74      0.65      0.69      1316
           1       0.87      0.91      0.89      3324

    accuracy                           0.84      4640
   macro avg       0.80      0.78      0.79      4640
weighted avg       0.83      0.84      0.83      4640



In [21]:
print(f" Accuracy: {acc}\n f1_score: {f1}")

 Accuracy: 0.8359913793103448
 f1_score: 0.8883017760164391


In [22]:
pickle.dump(model,open("fake_news_detection_model.pkl","wb"))
pickle.dump(vectorizer,open("vectorizer.pkl","wb"))